# Phân loại Email Spam
## 5. Tiền xử lý văn bản

### Quy trình tiền xử lý
1. Làm sạch văn bản thô (chuyển chữ thường, xóa thẻ HTML, chuẩn hóa số và dấu câu)
2. Tách từ, loại bỏ stop words, và chuẩn hóa từ về gốc (lemmatization)
3. Xây dựng bộ trích xuất đặc trưng TF-IDF từ đầu
4. Vector hóa tập huấn luyện (train) và tập kiểm thử (test)
5. Lưu trữ tất cả các kết quả trung gian lên đĩa

### Công thức tính TF-IDF

$$\text{TF}(t, d) = \frac{\text{số lần xuất hiện của } t \text{ trong } d}{\text{tổng số từ trong } d}$$

Với làm mịn phi tuyến (sublinear TF scaling): $\text{TF}_{\text{sub}}(t, d) = 1 + \log(\text{count}(t, d))$

$$\text{IDF}(t) = \log\!\left(\frac{N + 1}{df(t) + 1}\right) + 1 \quad \text{(đã làm mịn - smoothed)}$$

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Mỗi vector tài liệu sau đó được **chuẩn hóa L2** (L2 normalization) để loại bỏ ảnh hưởng của độ dài văn bản đến điểm tương đồng.


In [1]:
import numpy as np
import pandas as pd
import re, os, pickle, math, warnings
from collections import Counter
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split

df = pd.read_csv(r'./data/preprocessed/enron_cleaned.csv')
df['text'] = df['text'].fillna('')
print('Số lượng bản ghi đã tải từ dữ liệu đã làm sạch:', len(df))

[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1007)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1007)>


Số lượng bản ghi đã tải từ dữ liệu đã làm sạch: 33665


### 5.1 Hàm làm sạch văn bản

In [2]:
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Áp dụng pipeline tiền xử lý chuẩn NLP cho một tài liệu:
    Chuyển chữ thường, xóa thẻ HTML, xóa URL, chuẩn hóa số,
    lọc stop words và rút gọn từ về từ gốc (lemmatization).
    """
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)              # xóa HTML tags
    text = re.sub(r'http\S+|www\.\S+', '', text)      # xóa URLs
    text = re.sub(r'\b\d+\b', '__number__', text)     # chuẩn hóa số
    text = re.sub(r'!+', ' __exclamation__ ', text)   # chuẩn hóa dấu chấm than
    text = re.sub(r'\$+', ' __dollar__ ', text)       # chuẩn hóa dấu đô-la (tiền tệ)
    text = re.sub(r'[^a-z_\s]', ' ', text)            # xóa các ký tự đặc biệt còn lại
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in STOP_WORDS and len(t) > 1
    ]
    return ' '.join(tokens)

df['text_raw']   = df['text']
print('Đang thực hiện làm sạch dữ liệu văn bản...')
df['text_clean'] = df['text_raw'].apply(clean_text)
print('Làm sạch hoàn tất. Mẫu đầu ra:')
print(df['text_clean'].iloc[1][:300])

Đang thực hiện làm sạch dữ liệu văn bản...
Làm sạch hoàn tất. Mẫu đầu ra:
vastar resource inc gary production high island larger block __number__ __number__ commenced saturday __number__ __number__ __number__ __number__ gross carlos expects __number__ __number__ __number__ __number__ gross tomorrow vastar owns __number__ gross production george __number__ __number__ forwa


### 5.2 Chia tập Huấn luyện / Kiểm thử

In [3]:
X = df['text_clean'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Tập Train: {len(X_train):,} mẫu  |  Tập Test: {len(X_test):,} mẫu')
unique, counts = np.unique(y_train, return_counts=True)
print('Phân phối lớp trong tập Train:', dict(zip(unique, counts)))

Tập Train: 26,932 mẫu  |  Tập Test: 6,733 mẫu
Phân phối lớp trong tập Train: {'ham': np.int64(13236), 'spam': np.int64(13696)}


### 5.3 Bộ trích xuất đặc trưng TF-IDF — Viết từ đầu

Lớp cài đặt dưới đây mô phỏng chính xác hành vi của bộ `sklearn.feature_extraction.text.TfidfVectorizer` trong Scikit-learn.

**Các tham số hỗ trợ:**

| Tham số | Mô tả |
|-----------|-------------|
| `max_features` | Giữ lại tối đa K từ có tần suất xuất hiện cao nhất |
| `min_df` | Loại bỏ các từ xuất hiện trong ít hơn `min_df` văn bản |
| `max_df` | Loại bỏ các từ xuất hiện trong nhiều hơn `max_df` (tỷ lệ) văn bản |
| `sublinear_tf` | Thay thế TF thô bằng `1 + log(TF)` để giảm tác động của các từ lặp lại nhiều lần |
| `ngram_range` | Bộ giá trị `(min_n, max_n)` để sinh các cụm từ (n-grams) |


In [4]:
class TFIDFVectorizerFromScratch:
    """
    Bộ trích xuất đặc trưng TF-IDF viết từ đầu.

    Parameters
    ----------
    max_features : int hoặc None
        Số lượng đặc trưng tối đa giữ lại, được sắp xếp theo tần suất xuất hiện giảm dần.
    min_df : int
        Tần suất tài liệu tối thiểu để giữ lại từ khóa.
    max_df : float hoặc int
        Tần suất tài liệu tối đa để giữ lại từ khóa (tỷ lệ nếu là float, số lượng nếu là int).
    sublinear_tf : bool
        Áp dụng làm mịn phi tuyến TF: TF = 1 + log(count).
    ngram_range : tuple của (int, int)
        Dải n-gram để sinh đặc trưng.
    """

    def __init__(self, max_features=None, min_df=1,
                 max_df=1.0, sublinear_tf=False,
                 ngram_range=(1, 1)):
        self.max_features = max_features
        self.min_df       = min_df
        self.max_df       = max_df
        self.sublinear_tf = sublinear_tf
        self.ngram_range  = ngram_range
        self.vocabulary_  = {}  # term -> column index
        self.idf_         = {}  # term -> IDF value

    def _tokenize(self, text: str):
        """Sinh các unigram và n-gram từ văn bản đã được phân tách bằng khoảng trắng."""
        words = str(text).split()
        tokens = []
        min_n, max_n = self.ngram_range
        for n in range(min_n, max_n + 1):
            for i in range(len(words) - n + 1):
                tokens.append(' '.join(words[i:i + n]))
        return tokens

    def _compute_tf(self, tokens):
        """Tính toán tần suất từ (TF) cho một tài liệu."""
        counts = Counter(tokens)
        total  = len(tokens)
        if total == 0:
            return counts
        if self.sublinear_tf:
            return {t: 1 + math.log(c) for t, c in counts.items()} 
        return {t: c / total for t, c in counts.items()}

    def fit(self, corpus):
        """Học từ vựng và trọng số IDF từ tập dữ liệu huấn luyện."""
        N = len(corpus)
        df_counts = Counter()

        for text in corpus:
            unique_tokens = set(self._tokenize(text))
            for t in unique_tokens:
                df_counts[t] += 1

        # Áp dụng bộ lọc min_df / max_df
        max_df_abs = (
            self.max_df if isinstance(self.max_df, int)
            else int(self.max_df * N)
        )
        min_df_abs = self.min_df

        filtered = {
            t: c for t, c in df_counts.items()
            if min_df_abs <= c <= max_df_abs
        }

        # Tính toán IDF có làm mịn: log((N+1)/(df+1)) + 1
        idf_all = {
            t: math.log((N + 1) / (c + 1)) + 1
            for t, c in filtered.items()
        }

        # Sắp xếp các từ theo tần suất xuất hiện trên corpus (Document Frequency) giảm dần
        if self.max_features is not None:
            sorted_terms = sorted(filtered, key=filtered.get, reverse=True)
            sorted_terms = sorted_terms[:self.max_features]
        else:
            sorted_terms = sorted(filtered.keys())

        self.vocabulary_ = {t: i for i, t in enumerate(sorted_terms)}
        self.idf_        = {t: idf_all[t] for t in sorted_terms}
        return self

    def transform(self, corpus):
        """Chuyển đổi corpus thành ma trận đặc trưng TF-IDF."""
        V = len(self.vocabulary_)
        X = np.zeros((len(corpus), V), dtype=np.float32)

        for i, text in enumerate(corpus):
            tokens = self._tokenize(text)
            tf     = self._compute_tf(tokens)
            for term, tf_val in tf.items():
                if term in self.vocabulary_:
                    j       = self.vocabulary_[term]
                    X[i, j] = tf_val * self.idf_[term]

            # Chuẩn hóa L2
            norm = np.linalg.norm(X[i])
            if norm > 0:
                X[i] /= norm

        return X

    def fit_transform(self, corpus):
        return self.fit(corpus).transform(corpus)

print('Định nghĩa lớp TFIDFVectorizerFromScratch hoàn tất.')

Định nghĩa lớp TFIDFVectorizerFromScratch hoàn tất.


### 5.4 Huấn luyện bộ vector hóa và Chuyển đổi dữ liệu

In [5]:
print('Đang huấn luyện bộ vector hóa TFIDFVectorizerFromScratch trên tập huấn luyện...')
tfidf_scratch = TFIDFVectorizerFromScratch(
    max_features=5000, min_df=3, max_df=0.90,
    sublinear_tf=True, ngram_range=(1, 2)
)

X_train_vec = tfidf_scratch.fit_transform(X_train)
X_test_vec  = tfidf_scratch.transform(X_test)

print(f'Ma trận đặc trưng tập Train: {X_train_vec.shape}')
print(f'Ma trận đặc trưng tập Test : {X_test_vec.shape}')

# Hiển thị một số từ vựng cùng điểm IDF tương ứng
print('\nMẫu từ vựng học được (10 từ đầu tiên):')
for term in list(tfidf_scratch.vocabulary_.keys())[:10]:
    print(f'  {term:<35s}  IDF = {tfidf_scratch.idf_[term]:.4f}')

Đang huấn luyện bộ vector hóa TFIDFVectorizerFromScratch trên tập huấn luyện...
Ma trận đặc trưng tập Train: (26932, 5000)
Ma trận đặc trưng tập Test : (6733, 5000)

Mẫu từ vựng học được (10 từ đầu tiên):
  __number__                           IDF = 1.1828
  __number__ __number__                IDF = 1.4673
  __exclamation__                      IDF = 2.0118
  please                               IDF = 2.0167
  com                                  IDF = 2.2661
  subject                              IDF = 2.4269
  time                                 IDF = 2.4724
  enron                                IDF = 2.5108
  get                                  IDF = 2.5169
  __dollar__                           IDF = 2.5546


### 5.5 Lưu trữ các kết quả đặc trưng

In [6]:
DATA_DIR = r'./data/ready_for_train'
os.makedirs(DATA_DIR, exist_ok=True)

def save_pkl(obj, fname):
    with open(os.path.join(DATA_DIR, fname), 'wb') as f:
        pickle.dump(obj, f)

save_pkl(X_train_vec,   'X_train_tfidf.pkl')
save_pkl(X_test_vec,    'X_test_tfidf.pkl')
save_pkl(y_train,       'y_train.pkl')
save_pkl(y_test,        'y_test.pkl')
save_pkl(tfidf_scratch, 'tfidf_vectorizer.pkl')
print('Các tệp đặc trưng được lưu tại:', DATA_DIR)

Các tệp đặc trưng được lưu tại: ./data/ready_for_train
